# CREPE pitch-accuracy evaluation (MIR-1K)

Measures how accurately CREPE tracks pitch on **sung** audio, against
MIR-1K's hand-corrected ground-truth labels.

**Why evaluation rather than fine-tuning.** Pitch estimation is acoustic,
not linguistic — an F0 tracker doesn't care what language the vowel is in,
so there is little for Tagalog-specific training to improve. Fine-tuning
would also need frame-level F0 labels, which can't be hand-made; you'd need
a labelled corpus, which is what CREPE was already trained on.

**What the earlier notebook did wrong.** It ran `crepe.predict()` and saved
the output *as if it were ground truth*. Training on that is self-distillation:
the student can't beat the teacher, and the accuracy number is circular.
MIR-1K ships real labels (`.pv` files) — this notebook uses those.

**Output:** Raw Pitch Accuracy, Raw Chroma Accuracy, Voicing Recall and
Voicing False Alarm — the standard melody-extraction metrics, computed with
`mir_eval`, which is what published pitch-tracking papers report.

In [ ]:
#@title 1 — Install dependencies
!pip install -q crepe mir_eval soundfile resampy tensorflow pandas matplotlib

import numpy as np, pandas as pd, soundfile as sf, os, glob, json
import mir_eval, crepe
print('ready')

## 2 — Get MIR-1K

1000 sung clips (19 singers, Chinese karaoke) with per-frame pitch labels.
Language doesn't matter here: we're measuring F0 tracking, not words.

⚠️ **Dataset links move.** If the download below fails, get MIR-1K manually
(search "MIR-1K dataset Zenodo"), unzip it into your Drive, and set `ROOT`
to the folder that contains `Wavfile/` and `PitchLabel/`. Everything after
this cell only depends on `ROOT` being correct.

In [ ]:
#@title 2 — Dataset location
from google.colab import drive
drive.mount('/content/drive')

# Point this at the folder containing Wavfile/ and PitchLabel/
ROOT = '/content/drive/MyDrive/crepe_training/MIR-1K'  #@param {type:"string"}

WAV_DIR   = os.path.join(ROOT, 'Wavfile')
LABEL_DIR = os.path.join(ROOT, 'PitchLabel')

wavs = sorted(glob.glob(os.path.join(WAV_DIR, '*.wav')))
pvs  = sorted(glob.glob(os.path.join(LABEL_DIR, '*.pv')))
print(f'wav files   : {len(wavs)}')
print(f'label files : {len(pvs)}')
assert wavs and pvs, (
    'Not found. Set ROOT to the folder holding Wavfile/ and PitchLabel/.'
)

## 3 — Dataset sanity checks

Two things silently ruin this evaluation if assumed rather than verified:

**Channels.** MIR-1K wavs are stereo with *accompaniment in one channel and
the isolated voice in the other*. Mixing to mono feeds CREPE voice **plus**
music and understates its accuracy. We detect which channel is the vocal
rather than hardcoding it.

**Label hop size.** The `.pv` files are one pitch value per frame in MIDI
semitones (0 = unvoiced). If the hop is wrong, labels drift against the
audio and every metric collapses. We infer it from duration ÷ label count
and check it against the documented 20 ms.

In [ ]:
#@title 3 — Verify channels and label alignment
probe = wavs[0]
audio, sr = sf.read(probe)
print(f'{os.path.basename(probe)}  sr={sr}  shape={audio.shape}')

if audio.ndim == 2:
    # The vocal channel is the quieter, sparser one (it has silence between
    # phrases); the accompaniment plays continuously.
    energy = [float(np.mean(np.abs(audio[:, c]))) for c in range(audio.shape[1])]
    silence = [float(np.mean(np.abs(audio[:, c]) < 1e-3)) for c in range(audio.shape[1])]
    print(f'channel mean |x| : {energy}')
    print(f'fraction silent  : {silence}')
    VOCAL_CH = int(np.argmax(silence))
    print(f'--> using channel {VOCAL_CH} as the voice')
else:
    VOCAL_CH = None
    print('mono file — no channel choice to make')

lab = np.loadtxt(os.path.join(LABEL_DIR,
                 os.path.basename(probe).replace('.wav', '.pv')))
dur = len(audio) / sr
inferred_hop_ms = dur / len(lab) * 1000
print(f'\nduration {dur:.2f}s, {len(lab)} labels '
      f'-> inferred hop {inferred_hop_ms:.1f} ms')
print(f'labels: min={lab.min():.1f} max={lab.max():.1f} semitones, '
      f'{np.mean(lab == 0) * 100:.0f}% unvoiced')

HOP_MS = 20.0
if abs(inferred_hop_ms - HOP_MS) > 2:
    print(f'\n!! inferred hop differs from {HOP_MS} ms — using inferred value')
    HOP_MS = inferred_hop_ms

In [ ]:
#@title 4 — Evaluation function
def semitones_to_hz(semi):
    """MIR-1K labels are MIDI semitones, 0 meaning unvoiced.
    mir_eval wants Hz with non-positive = unvoiced."""
    hz = np.zeros_like(semi, dtype=float)
    voiced = semi > 0
    hz[voiced] = 440.0 * 2 ** ((semi[voiced] - 69.0) / 12.0)
    return hz


def evaluate_clip(wav_path, confidence_threshold=0.5, model_capacity='full'):
    """Runs CREPE on one clip and scores it against the ground truth.

    mir_eval resamples the estimate onto the reference timebase itself, so
    CREPE's 10 ms grid and the label's 20 ms grid don't need to match.
    """
    audio, sr = sf.read(wav_path)
    if audio.ndim == 2:
        audio = audio[:, VOCAL_CH]

    label_path = os.path.join(
        LABEL_DIR, os.path.basename(wav_path).replace('.wav', '.pv'))
    ref_semi = np.loadtxt(label_path)
    ref_freq = semitones_to_hz(ref_semi)
    ref_time = np.arange(len(ref_freq)) * (HOP_MS / 1000.0)

    est_time, est_freq, confidence, _ = crepe.predict(
        audio, sr, model_capacity=model_capacity,
        viterbi=True, step_size=10, verbose=0)

    # Below-threshold frames are the model saying "no reliable pitch here" —
    # scored as unvoiced rather than as a wrong note.
    est_freq = est_freq.copy()
    est_freq[confidence < confidence_threshold] = 0.0

    scores = mir_eval.melody.evaluate(ref_time, ref_freq, est_time, est_freq)
    scores['clip'] = os.path.basename(wav_path)
    scores['mean_confidence'] = float(np.mean(confidence))
    return scores


print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v)
                  for k, v in evaluate_clip(wavs[0]).items()}, indent=2))

In [ ]:
#@title 5 — Run the evaluation
N_CLIPS = 100  #@param {type:"integer"}
CONFIDENCE_THRESHOLD = 0.5  #@param {type:"number"}
MODEL_CAPACITY = 'full'  #@param ['tiny','small','medium','large','full']

# Evenly spaced rather than the first N, so the sample spans all singers
# instead of whoever sorts first alphabetically.
idx = np.linspace(0, len(wavs) - 1, min(N_CLIPS, len(wavs))).astype(int)
subset = [wavs[i] for i in idx]

rows, failed = [], []
for i, w in enumerate(subset, 1):
    try:
        rows.append(evaluate_clip(w, CONFIDENCE_THRESHOLD, MODEL_CAPACITY))
    except Exception as exc:
        failed.append((os.path.basename(w), str(exc)))
    if i % 10 == 0:
        print(f'{i}/{len(subset)}')

df = pd.DataFrame(rows)
df.to_csv('/content/drive/MyDrive/crepe_eval_results.csv', index=False)
print(f'\nevaluated {len(df)} clips, {len(failed)} failed')
for name, err in failed[:5]:
    print('  ', name, err[:80])

In [ ]:
#@title 6 — Results table (this is what goes in the thesis)
metrics = ['Raw Pitch Accuracy', 'Raw Chroma Accuracy',
           'Voicing Recall', 'Voicing False Alarm', 'Overall Accuracy']

summary = pd.DataFrame({
    'metric': metrics,
    'mean':   [df[m].mean() for m in metrics],
    'std':    [df[m].std()  for m in metrics],
    'median': [df[m].median() for m in metrics],
})
summary[['mean', 'std', 'median']] *= 100
print(f'CREPE "{MODEL_CAPACITY}", confidence >= {CONFIDENCE_THRESHOLD}, '
      f'n = {len(df)} clips\n')
print(summary.round(2).to_string(index=False))
print('\nRaw Pitch Accuracy  = % of voiced frames within 50 cents of truth')
print('Raw Chroma Accuracy = same, ignoring octave errors')
print('Voicing False Alarm = % of silent frames wrongly called voiced (lower is better)')

In [ ]:
#@title 7 — Threshold sweep: justify the app's confidence cutoff
# The app uses kDefaultPitchConfidenceThreshold = 0.15, lowered from 0.5
# during debugging. This turns that choice into a measured one: too high and
# real singing is dropped (voicing recall falls), too low and noise is called
# pitch (false alarm rises).
sweep_clips = subset[:30]
out = []
for th in [0.05, 0.10, 0.15, 0.25, 0.50, 0.70]:
    r = [evaluate_clip(w, th, MODEL_CAPACITY) for w in sweep_clips]
    d = pd.DataFrame(r)
    out.append({
        'threshold':   th,
        'RPA':         d['Raw Pitch Accuracy'].mean() * 100,
        'VoicingRec':  d['Voicing Recall'].mean() * 100,
        'VoicingFA':   d['Voicing False Alarm'].mean() * 100,
        'Overall':     d['Overall Accuracy'].mean() * 100,
    })
    print(f"threshold {th}: done")

print()
print(pd.DataFrame(out).round(2).to_string(index=False))

In [ ]:
#@title 8 — Plot
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['Raw Pitch Accuracy'] * 100, bins=20,
             color='#00b4d8', edgecolor='white')
axes[0].axvline(df['Raw Pitch Accuracy'].mean() * 100, color='crimson',
                linestyle='--',
                label=f"mean {df['Raw Pitch Accuracy'].mean()*100:.1f}%")
axes[0].set_xlabel('Raw Pitch Accuracy (%)')
axes[0].set_ylabel('clips')
axes[0].set_title('Per-clip pitch accuracy')
axes[0].legend()

axes[1].scatter(df['mean_confidence'], df['Raw Pitch Accuracy'] * 100,
                alpha=0.6, color='#0077b6')
axes[1].set_xlabel('mean CREPE confidence')
axes[1].set_ylabel('Raw Pitch Accuracy (%)')
axes[1].set_title('Does confidence predict accuracy?')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/crepe_eval.png', dpi=150)
plt.show()

## 9 — Optional: check the model the app actually ships

Everything above scores the **reference** CREPE. Your app runs
`assets/models/huni_crepe.tflite` on-device, which is a different artifact —
and an export/quantisation step can silently degrade a model.

Upload that file to Drive and run this to confirm the shipped model behaves
like the reference. If the two disagree, the problem is the export, not CREPE.

The app feeds 1024-sample frames and applies DC removal + z-score
normalisation — matched here deliberately, because preprocessing mismatch is
exactly the bug class that cost time earlier (a stray pre-emphasis filter).

In [ ]:
#@title 9 — Compare huni_crepe.tflite against reference CREPE
import tensorflow as tf

TFLITE = '/content/drive/MyDrive/crepe_training/huni_crepe.tflite'  #@param {type:"string"}

interp = tf.lite.Interpreter(model_path=TFLITE)
interp.allocate_tensors()
inp, outp = interp.get_input_details()[0], interp.get_output_details()[0]
print('input ', inp['shape'], inp['dtype'])
print('output', outp['shape'], outp['dtype'])

CENTS = np.linspace(0, 7180, 360) + 1997.3794084376191  # CREPE's bin centres

def tflite_pitch(frame):
    """One 1024-sample frame -> (Hz, confidence), mirroring the app."""
    x = frame - frame.mean()
    s = x.std()
    x = x / s if s > 1e-8 else x
    interp.set_tensor(inp['index'], x.reshape(inp['shape']).astype(np.float32))
    interp.invoke()
    act = interp.get_tensor(outp['index']).flatten()
    peak = int(np.argmax(act))
    lo, hi = max(0, peak - 4), min(len(act), peak + 5)
    w = act[lo:hi]
    cents = float(np.sum(CENTS[lo:hi] * w) / np.sum(w)) if w.sum() > 0 else 0.0
    return 10.0 * 2 ** (cents / 1200.0), float(act[peak])

audio, sr = sf.read(subset[0])
if audio.ndim == 2:
    audio = audio[:, VOCAL_CH]

hop = int(sr * 0.01)
tf_hz, tf_conf, times = [], [], []
for start in range(0, len(audio) - 1024, hop):
    hz, cf = tflite_pitch(audio[start:start + 1024])
    tf_hz.append(hz); tf_conf.append(cf); times.append(start / sr)

ref_t, ref_hz, ref_cf, _ = crepe.predict(audio, sr, viterbi=False,
                                         step_size=10, verbose=0)
n = min(len(tf_hz), len(ref_hz))
both = (np.array(tf_conf[:n]) > 0.5) & (ref_cf[:n] > 0.5)
if both.sum():
    cents_err = 1200 * np.log2(np.array(tf_hz[:n])[both] / ref_hz[:n][both])
    print(f'\nframes compared      : {both.sum()}')
    print(f'median abs error     : {np.median(np.abs(cents_err)):.1f} cents')
    print(f'within 50 cents      : {np.mean(np.abs(cents_err) < 50)*100:.1f}%')
    print(f'mean confidence tflite={np.mean(tf_conf):.3f} ref={np.mean(ref_cf):.3f}')
    print('\n>50 cents median means the exported model does NOT match reference '
          'CREPE — investigate the export, not the algorithm.')
else:
    print('No frames where both models were confident — check preprocessing.')